In [5]:
import pandas as pd
import mido
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.neighbors import KNeighborsRegressor
from rich.console import Console

console = Console()


In [6]:
# some midi files use velocity 0 as note off
# to fix import to another program (ableton) and re-export
midi_file = mido.MidiFile('training_files/beethoven.mid')
df = pd.DataFrame([
    message.dict() for message in midi_file.tracks[0] if not message.is_meta
])

df.to_csv('test.csv')
lookback = 7
midi_file



MidiFile(type=0, ticks_per_beat=96, tracks=[
  MidiTrack([
    MetaMessage('track_name', name='No Name', time=0),
    MetaMessage('time_signature', numerator=2, denominator=4, clocks_per_click=36, notated_32nd_notes_per_beat=8, time=0),
    Message('program_change', channel=0, program=0, time=0),
    Message('control_change', channel=0, control=0, value=0, time=0),
    Message('control_change', channel=0, control=32, value=0, time=0),
    Message('pitchwheel', channel=0, pitch=0, time=0),
    Message('control_change', channel=0, control=1, value=0, time=0),
    Message('control_change', channel=0, control=7, value=127, time=0),
    Message('control_change', channel=0, control=10, value=73, time=0),
    Message('control_change', channel=0, control=64, value=0, time=0),
    Message('control_change', channel=0, control=91, value=91, time=0),
    Message('control_change', channel=0, control=93, value=18, time=0),
    Message('note_on', channel=0, note=52, velocity=100, time=0),
    Message

In [7]:
wrangled_df = df[df['type'] != 'sysex']
wrangled_df = wrangled_df[wrangled_df['type'] != 'program_change']
type_names = ['note_on', 'note_off', 'control_change']
cat_type = pd.CategoricalDtype(type_names, ordered=True)
wrangled_df['type_code'] = df['type'].astype(cat_type).cat.codes
wrangled_df = wrangled_df[['type_code', 'time', 'note', 'velocity', 'control', 'value']]
wrangled_df = wrangled_df[~((wrangled_df['control'] == 0) | (wrangled_df['control'] == 1))]
wrangled_df.to_csv('wrangled_df.csv')

param_num = wrangled_df.shape[1]

filled_df = wrangled_df.ffill()
filled_df = filled_df.bfill()
filled_df = filled_df.reset_index(drop=True)

lookback_df = filled_df.add_suffix(f'_{lookback}')

for x in range(lookback):
    filled_df
    lookback_df = lookback_df.join(filled_df.shift(-x).add_suffix(f'_{lookback - 1 - x}'))
    lookback_df = lookback_df.iloc[0:(1 - lookback), :]


/var/folders/33/wgwlkyg527v5hnsfct930frm0000gn/T/ipykernel_74535/1192573765.py:5: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  wrangled_df['type_code'] = df['type'].astype(cat_type).cat.codes


In [8]:
data = lookback_df.loc[:, ~lookback_df.columns.str.endswith('_0')].copy()
target = lookback_df['type_code_0'].copy()



X_train_type, X_test_type, y_train_type, y_test_type = train_test_split(data, target, random_state=0)

type_knn = KNeighborsClassifier(n_neighbors=6).fit(X_train_type, y_train_type)
type_lr = LogisticRegression(C=0.01, max_iter=5000, solver='saga').fit(X_train_type, y_train_type)
type_linear_svm = CalibratedClassifierCV(LinearSVC(C=0.01, dual=False, max_iter=5000)).fit(X_train_type, y_train_type)

print(f'knn training score: {type_knn.score(X_train_type, y_train_type):.2f}')
print(f'knn test score: {type_knn.score(X_test_type, y_test_type):.2f}\n')
print(f'lr training score: {type_lr.score(X_train_type, y_train_type):.2f}')
print(f'lr test score: {type_lr.score(X_test_type, y_test_type):.2f}\n')
print(f'linear svm training score: {type_linear_svm.score(X_train_type, y_train_type):.2f}')
print(f'linear svm test score: {type_linear_svm.score(X_test_type, y_test_type):.2f}\n')


knn training score: 0.85
knn test score: 0.80

lr training score: 0.68
lr test score: 0.70

linear svm training score: 0.68
linear svm test score: 0.70



In [9]:
data = lookback_df.loc[:, ~lookback_df.columns.str.endswith('_0')].copy()
data['type_code_0'] = lookback_df['type_code_0'].copy()
target = lookback_df['note_0'].copy()



X_train_note, X_test_note, y_train_note, y_test_note = train_test_split(data, target, random_state=0)

note_knn = KNeighborsClassifier(n_neighbors=1).fit(X_train_note, y_train_note)
note_lr = LogisticRegression(C=0.0001, max_iter=5000, solver='saga').fit(X_train_note, y_train_note)
note_linear_svm = CalibratedClassifierCV(LinearSVC(C=0.0001, dual=False, max_iter=5000)).fit(X_train_note, y_train_note)

print(f'knn training score: {note_knn.score(X_train_note, y_train_note):.2f}')
print(f'knn test score: {note_knn.score(X_test_note, y_test_note):.2f}\n')
print(f'lr training score: {note_lr.score(X_train_note, y_train_note):.2f}')
print(f'lr test score: {note_lr.score(X_test_note, y_test_note):.2f}\n')
print(f'linear svm training score: {note_linear_svm.score(X_train_note, y_train_note):.2f}')
print(f'linear svm test score: {note_linear_svm.score(X_test_note, y_test_note):.2f}\n')


/Users/diegomoca/Documents/01_proyectos/pais-de-jauja/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


knn training score: 1.00
knn test score: 0.41

lr training score: 0.11
lr test score: 0.10

linear svm training score: 0.10
linear svm test score: 0.09



In [10]:
data_velocity = lookback_df.loc[:, ~lookback_df.columns.str.endswith('_0')].copy()
data_velocity[['type_code_0', 'note_0']] = lookback_df[['type_code_0', 'note_0']].copy()
target_velocity = lookback_df['velocity_0'].copy()



X_train_velocity, X_test_velocity, y_train_velocity, y_test_velocity = train_test_split(data_velocity, target_velocity, random_state=0)

velocity_knn = KNeighborsClassifier(n_neighbors=1).fit(X_train_velocity, y_train_velocity)
velocity_lr = LogisticRegression(C=0.0001, max_iter=5000, solver='saga').fit(X_train_velocity, y_train_velocity)
velocity_linear_svm = CalibratedClassifierCV(LinearSVC(C=0.0001, dual=False, max_iter=5000)).fit(X_train_velocity, y_train_velocity)

print(f'knn training score: {velocity_knn.score(X_train_velocity, y_train_velocity):.2f}')
print(f'knn test score: {velocity_knn.score(X_test_velocity, y_test_velocity):.2f}\n')
print(f'lr training score: {velocity_lr.score(X_train_velocity, y_train_velocity):.2f}')
print(f'lr test score: {velocity_lr.score(X_test_velocity, y_test_velocity):.2f}\n')
print(f'linear svm training score: {velocity_linear_svm.score(X_train_velocity, y_train_velocity):.2f}')
print(f'linear svm test score: {velocity_linear_svm.score(X_test_velocity, y_test_velocity):.2f}\n')


/Users/diegomoca/Documents/01_proyectos/pais-de-jauja/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


knn training score: 1.00
knn test score: 0.55

lr training score: 0.50
lr test score: 0.51

linear svm training score: 0.52
linear svm test score: 0.53



In [11]:
data = lookback_df.loc[:, ~lookback_df.columns.str.endswith('_0')].copy()
data['type_code_0'] = lookback_df['type_code_0'].copy()
target = lookback_df['control_0'].copy()



X_train, X_test, y_train, y_test = train_test_split(data, target, random_state=0)

control_knn = KNeighborsClassifier(n_neighbors=8).fit(X_train, y_train)
control_lr = LogisticRegression(C=1, max_iter=5000, solver='saga').fit(X_train, y_train)
control_linear_svm_l2 = LinearSVC(C=1, dual=False, max_iter=5000).fit(X_train, y_train)
control_linear_svm_l1 = LinearSVC(penalty='l1', dual=False, C=1, max_iter=5000).fit(X_train, y_train)


print(f'knn training score: {control_knn.score(X_train, y_train):.2f}')
print(f'knn test score: {control_knn.score(X_test, y_test):.2f}\n')
print(f'lr training score: {control_lr.score(X_train, y_train):.2f}')
print(f'lr test score: {control_lr.score(X_test, y_test):.2f}\n')
print(f'linear svm l2 training score: {control_linear_svm_l2.score(X_train, y_train):.2f}')
print(f'linear svm l2 test score: {control_linear_svm_l2.score(X_test, y_test):.2f}\n')
print(f'linear svm l1 training score: {control_linear_svm_l1.score(X_train, y_train):.2f}')
print(f'linear svm l1 test score: {control_linear_svm_l1.score(X_test, y_test):.2f}\n')


/Users/diegomoca/Documents/01_proyectos/pais-de-jauja/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


knn training score: 1.00
knn test score: 1.00

lr training score: 1.00
lr test score: 1.00

linear svm l2 training score: 1.00
linear svm l2 test score: 1.00

linear svm l1 training score: 1.00
linear svm l1 test score: 1.00



/Users/diegomoca/Documents/01_proyectos/pais-de-jauja/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [12]:
data = lookback_df.loc[:, ~lookback_df.columns.str.endswith('_0')].copy()
data[['type_code_0', 'control_0']] = lookback_df[['type_code_0', 'control_0']].copy()
target = lookback_df['value_0'].copy()



X_train, X_test, y_train, y_test = train_test_split(data, target, random_state=0)

value_knn = KNeighborsClassifier(n_neighbors=8).fit(X_train, y_train)
value_lr = LogisticRegression(C=0.00001, max_iter=5000, solver='saga').fit(X_train, y_train)
value_linear_svm_l2 = LinearSVC(C=0.00001, dual=False, max_iter=5000).fit(X_train, y_train)
value_linear_svm_l1 = LinearSVC(penalty='l1', dual=False, C=0.00001, max_iter=5000).fit(X_train, y_train)


print(f'knn training score: {value_knn.score(X_train, y_train):.2f}')
print(f'knn test score: {value_knn.score(X_test, y_test):.2f}\n')
print(f'lr training score: {value_lr.score(X_train, y_train):.2f}')
print(f'lr test score: {value_lr.score(X_test, y_test):.2f}\n')
print(f'linear svm l2 training score: {value_linear_svm_l2.score(X_train, y_train):.2f}')
print(f'linear svm l2 test score: {value_linear_svm_l2.score(X_test, y_test):.2f}\n')
print(f'linear svm l1 training score: {value_linear_svm_l1.score(X_train, y_train):.2f}')
print(f'linear svm l1 test score: {value_linear_svm_l1.score(X_test, y_test):.2f}\n')


knn training score: 1.00
knn test score: 0.99

lr training score: 0.99
lr test score: 0.99

linear svm l2 training score: 0.99
linear svm l2 test score: 0.99

linear svm l1 training score: 0.99
linear svm l1 test score: 0.99



/Users/diegomoca/Documents/01_proyectos/pais-de-jauja/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [13]:
data = lookback_df.drop('time_0', axis=1).copy()
target = lookback_df['time_0'].copy()



X_train_time, X_test_time, y_train_time, y_test_time = train_test_split(data, target, random_state=0)

time_knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_time, y_train_time)
time_lr = LogisticRegression(C=0.0001, max_iter=5000, solver='saga').fit(X_train_time, y_train_time)
time_linear_svm = CalibratedClassifierCV(LinearSVC(C=0.00001, dual=False, max_iter=5000)).fit(X_train_time, y_train_time)

print(f'knn training score: {time_knn.score(X_train_time, y_train_time):.2f}')
print(f'knn test score: {time_knn.score(X_test_time, y_test_time):.2f}\n')
print(f'lr training score: {time_lr.score(X_train_time, y_train_time):.2f}')
print(f'lr test score: {time_lr.score(X_test_time, y_test_time):.2f}\n')
print(f'linear svm training score: {time_linear_svm.score(X_train_time, y_train_time):.2f}')
print(f'linear svm test score: {time_linear_svm.score(X_test_time, y_test_time):.2f}\n')


/Users/diegomoca/Documents/01_proyectos/pais-de-jauja/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


knn training score: 0.83
knn test score: 0.77

lr training score: 0.66
lr test score: 0.65

linear svm training score: 0.66
linear svm test score: 0.65



In [16]:
midi_file = mido.MidiFile(ticks_per_beat=96)
midi_track = mido.MidiTrack()
midi_file.tracks.append(midi_track)
generated_msgs = []

# i = 0
# random = np.random.randint(1, 5)
X_start = lookback_df.loc[:0, ~lookback_df.columns.str.endswith('_0')].copy()
for x in range(1000):
    # if i == random:
    #     X_start = lookback_df.loc[[np.random.choice(lookback_df.shape[0])], ~lookback_df.columns.str.endswith('_0')]
    #     random = np.random.randint(1, 20)
    #     i = 0
    # i += 1
    print(f'iteration {x}') 
    X_new = X_start.copy()
    new_type = type_linear_svm.predict(X_new).squeeze()

    # TODO add probability of prediction besides choice
    if np.random.choice([0, 1], p=[0.7, 0.3]) == 0:
        if np.random.randint(0, 1) == 0:
            new_type = type_lr.predict(X_new)
        else:
            new_type = type_linear_svm.predict(X_new)
    else:
        if np.random.randint(0, 1) == 0:
            new_type = np.random.choice(type_lr.classes_, p=type_lr.predict_proba(X_new)[0])
        else:
            new_type = np.random.choice(type_linear_svm.classes_, p=type_linear_svm.predict_proba(X_new)[0])
    X_new['type_code_0'] = new_type

    new_control = control_knn.predict(X_new).squeeze()
    X_new['control_0'] = new_control

    new_value = value_linear_svm_l2.predict(X_new).squeeze()
    X_new.drop('control_0', axis=1, inplace=True)
    
    if new_type == 0:
        if np.random.choice([0, 1], p=[0.7, 0.3]) == 0:
            if np.random.randint(0, 1) == 0:
                new_note = note_lr.predict(X_new)
            else:
                new_note = note_linear_svm.predict(X_new)
        else:
            if np.random.randint(0, 1) == 0:
                new_note = np.random.choice(note_lr.classes_, p=note_lr.predict_proba(X_new)[0])
            else:
                new_note = np.random.choice(note_linear_svm.classes_, p=note_linear_svm.predict_proba(X_new)[0])
    else:
        new_note = note_knn.predict(X_new).squeeze()
    X_new['note_0'] = new_note

    if np.random.choice([0, 1], p=[0.7, 0.3]) == 0:
        if np.random.randint(0, 1) == 0:
            new_velocity = velocity_lr.predict(X_new)
        else:
            new_velocity = velocity_linear_svm.predict(X_new)
    else:
        if np.random.randint(0, 1) == 0:
            new_velocity = np.random.choice(velocity_lr.classes_, p=velocity_lr.predict_proba(X_new)[0])
        else:
            new_velocity = np.random.choice(velocity_linear_svm.classes_, p=velocity_linear_svm.predict_proba(X_new)[0])
    if new_velocity < 0:
        new_velocity = 0
    X_new['velocity_0'] = new_velocity
    X_new['control_0']  = new_control
    X_new['value_0']    = new_value

    if np.random.choice([0, 1], p=[0.7, 0.3]) == 0:
        if np.random.randint(0, 1) == 0:
            new_time = time_lr.predict(X_new)
        else:
            new_time = time_linear_svm.predict(X_new)
    else:
        if np.random.randint(0, 1) == 0:
            new_time = np.random.choice(time_lr.classes_, p=time_lr.predict_proba(X_new)[0])
        else:
            new_time = np.random.choice(time_linear_svm.classes_, p=time_linear_svm.predict_proba(X_new)[0])

    X_new['time_0'] = new_time

    
    # drop last set of columns and replace the suffixes
    # reorder new message
    X_new = X_new[['type_code_0', 'time_0', 'note_0', 'velocity_0', 'control_0', 'value_0']]
    # drop last set of columns
    X_start = X_start.loc[:0, ~X_start.columns.str.endswith(f'_{lookback}')].copy()
    # append new message to old ones for next cycle
    X_start = pd.concat([X_start, X_new], axis=1)
    # remove all suffixes
    for count in range(lookback):
        X_start.columns = X_start.columns.str.removesuffix(f'_{lookback - count - 1}').copy()
        # insert new suffixes
        index = count * param_num
        for param in range(param_num):
            X_start.rename(
                columns={
                    X_start.columns[index + param]: f'{X_start.columns[index + param]}_{lookback - (index // param_num)}'
                }, inplace = True
            )
    # filter generated message and remove its suffixes
    X_new = X_new.loc[:, X_new.columns.str.endswith('_0')].copy()
    X_new.columns = X_new.columns.str.removesuffix('_0')


    new_msg = X_new.round(decimals=0).astype('int')
    new_msg['type'] = type_names[new_msg['type_code'].iloc[-1]]
    new_msg.drop('type_code', axis=1, inplace=True)

    new_msg = new_msg.to_dict(orient='records')[0]
    if new_msg['type'] in ['note_on', 'note_off']:
        new_msg.pop('control', None)
        new_msg.pop('value', None)
    elif new_msg['type'] == 'control_change':
        new_msg.pop('note', None)
        new_msg.pop('velocity', None)


    generated_msgs.append(new_msg.copy())

    midi_track.append(mido.Message(**new_msg))

midi_file.save('b.mid')
df_generated = pd.DataFrame(generated_msgs)
df_generated.to_csv('generated_messages.csv')


iteration 0
iteration 1
iteration 2
iteration 3
iteration 4
iteration 5
iteration 6
iteration 7
iteration 8
iteration 9
iteration 10
iteration 11
iteration 12
iteration 13
iteration 14
iteration 15
iteration 16
iteration 17
iteration 18
iteration 19
iteration 20
iteration 21
iteration 22
iteration 23
iteration 24
iteration 25
iteration 26
iteration 27
iteration 28
iteration 29
iteration 30
iteration 31
iteration 32
iteration 33
iteration 34
iteration 35
iteration 36
iteration 37
iteration 38
iteration 39
iteration 40
iteration 41
iteration 42
iteration 43
iteration 44
iteration 45
iteration 46
iteration 47
iteration 48
iteration 49
iteration 50
iteration 51
iteration 52
iteration 53
iteration 54
iteration 55
iteration 56
iteration 57
iteration 58
iteration 59
iteration 60
iteration 61
iteration 62
iteration 63
iteration 64
iteration 65
iteration 66
iteration 67
iteration 68
iteration 69
iteration 70
iteration 71
iteration 72
iteration 73
iteration 74
iteration 75
iteration 76
iteration